# ECRTM v2 - Optimisation de la vitesse

## Hypothèse
Cette variante cherche a réduire le temps d'entraînement d'ECRTM sans changer les métriques finales (C_V, TD, Purity, NMI).

## Intuition
Le gain vient de l'implementation: mixed precision (AMP), transferts non bloquants vers GPU, DataLoader optimise et early stopping.

## Plan du notebook
1. Charger la même base que ECRTM baseline.
2. Entrainer avec une boucle acceleree mais même modèle ECRTM.
3. Exporter les mêmes sorties `.mat` + timings uniformes.
4. Evaluer avec les mêmes métriques pour une comparaison propre.


In [1]:
import sys
!{sys.executable} -m pip install -q pyyaml scipy pandas scikit-learn tqdm torch gensim


In [2]:
import os
import time
import sys
import subprocess
from contextlib import nullcontext
from pathlib import Path
from types import SimpleNamespace

import yaml
import numpy as np
import pandas as pd
import scipy.io
import scipy.sparse as sp
import torch
from torch.cuda.amp import autocast, GradScaler  # backward compatibility
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel
from sklearn.metrics import normalized_mutual_info_score
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm


IS_KAGGLE = Path('/kaggle').exists()

MODEL_TAG = 'ECRTM_v2_SPEED'

SPEED_TUNING = {
    # Adaptive AMP: try AMP first for speed, fallback to full precision if unstable.
    'use_amp': True,
    'amp_adaptive_fallback': True,
    'prefer_bf16': True,

    'num_workers': 2,
    'pin_memory': True,
    'non_blocking_transfer': True,
    'grad_clip': 1.0,

    # Keep disabled by default to preserve baseline-level quality.
    'use_early_stopping': False,
    'early_stopping_patience': 12,
    'early_stopping_min_delta': 1e-4,
    'min_epochs_before_stop': 80,
    'restore_best_state': True,

    # Safety for NaN/Inf loss handling
    'max_non_finite_batches': 8,
}


def _unique_paths(paths):
    uniq = []
    seen = set()
    for p in paths:
        try:
            rp = Path(p).resolve()
        except Exception:
            continue
        key = str(rp)
        if key not in seen:
            seen.add(key)
            uniq.append(rp)
    return uniq


def detect_project_root():
    # Kaggle may mount datasets separately (no single project_root/data directory).
    if IS_KAGGLE:
        return Path('/kaggle/working')

    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / 'ECTRM_source' / 'ECRTM' / 'data').exists():
            return candidate
        if (candidate / 'Projet_PPD' / 'Projet_PPD').exists():
            return candidate / 'Projet_PPD' / 'Projet_PPD'
        if (candidate / 'Projet_PPD').exists():
            return candidate / 'Projet_PPD'

    return cwd

PROJECT_ROOT = detect_project_root()
DATA_ROOT = PROJECT_ROOT / 'data'

if IS_KAGGLE:
    OUT_DIR_ECRTM = Path('/kaggle/working') / 'Sortie_mat' / 'output' / 'kaggle' / MODEL_TAG
else:
    OUT_DIR_ECRTM = PROJECT_ROOT / 'output' / MODEL_TAG
OUT_DIR_ECRTM.mkdir(parents=True, exist_ok=True)


def detect_ecrtm_repo(project_root):
    cwd = Path.cwd().resolve()
    candidates = [
        project_root,
        project_root.parent,
        project_root / 'ECTRM_source' / 'ECRTM',
        project_root / 'ECTRM_source' / 'ECRTM' / 'ECRTM',
        project_root.parent / 'ECTRM_source' / 'ECRTM',
        project_root.parent / 'ECTRM_source' / 'ECRTM' / 'ECRTM',
        project_root.parent.parent / 'ECTRM_source' / 'ECRTM',
        project_root.parent.parent / 'ECTRM_source' / 'ECRTM' / 'ECRTM',
        cwd / 'ECTRM_source' / 'ECRTM',
        cwd / 'ECTRM_source' / 'ECRTM' / 'ECRTM',
        cwd.parent / 'ECTRM_source' / 'ECRTM',
        cwd.parent / 'ECTRM_source' / 'ECRTM' / 'ECRTM',
    ]

    if IS_KAGGLE:
        input_root = Path('/kaggle/input')
        if input_root.exists():
            for top in input_root.iterdir():
                if not top.is_dir():
                    continue
                candidates.extend([
                    top / 'ECTRM_source' / 'ECRTM',
                    top / 'ECTRM_source' / 'ECRTM' / 'ECRTM',
                    top / 'PPD_2026_ECTRM' / 'ECTRM_source' / 'ECRTM',
                    top / 'PPD_2026_ECTRM' / 'ECTRM_source' / 'ECRTM' / 'ECRTM',
                    top / 'ECRTM',
                ])

    for candidate in _unique_paths(candidates):
        if (candidate / 'models').exists() and (candidate / 'configs').exists():
            return candidate

    # Fallback: recursive search in common roots (handles unusual Kaggle input names).
    search_roots = [Path('/kaggle/input'), project_root, cwd] if IS_KAGGLE else [project_root, cwd]
    seen = set()
    for base in search_roots:
        if not base.exists():
            continue
        for root, _, _ in os.walk(base):
            rp = Path(root)
            key = str(rp)
            if key in seen:
                continue
            seen.add(key)
            if (rp / 'models').exists() and (rp / 'configs').exists():
                return rp

    return None


def clone_ecrtm_repo_kaggle():
    if not IS_KAGGLE:
        return None

    clone_root = Path('/kaggle/working') / '_ecrtm_source'
    repo_dir = clone_root / 'ecrtm'
    clone_root.mkdir(parents=True, exist_ok=True)

    if not repo_dir.exists():
        print('ECRTM source introuvable localement, tentative de clone GitHub...')
        try:
            subprocess.check_call([
                'git', 'clone', '--depth', '1',
                'https://github.com/bobxwu/ecrtm',
                str(repo_dir),
            ])
        except Exception as e:
            print(f'Clone GitHub impossible: {e}')
            return None

    for candidate in [repo_dir / 'ECRTM', repo_dir]:
        if (candidate / 'models').exists() and (candidate / 'configs').exists():
            return candidate

    return None


ECRTM_REPO = detect_ecrtm_repo(PROJECT_ROOT)
if ECRTM_REPO is None:
    ECRTM_REPO = clone_ecrtm_repo_kaggle()
if ECRTM_REPO is None:
    raise FileNotFoundError(
        'Impossible de localiser ECRTM (models/ + configs/). '
        'Ajoutez ECTRM_source/ECRTM dans les inputs Kaggle, ou activez Internet pour le clone auto.'
    )

if str(ECRTM_REPO) not in sys.path:
    sys.path.append(str(ECRTM_REPO))

from models.ECRTM import ECRTM

CONFIGS = {
    '20NG': ECRTM_REPO / 'configs' / 'model' / 'ECRTM_20NG.yaml',
    'AGNews': ECRTM_REPO / 'configs' / 'model' / 'ECRTM_AGNews.yaml',
    'IMDB': ECRTM_REPO / 'configs' / 'model' / 'ECRTM_IMDB.yaml',
    'YahooAnswer': ECRTM_REPO / 'configs' / 'model' / 'ECRTM_YahooAnswer.yaml',
}

TARGET_DATASETS = ['20NG', 'AGNews', 'IMDB', 'YahooAnswer']
K_VALUES = [20, 50, 100]
SEED = 42
FORCE_RECOMPUTE_METRICS = True

print(f'IS_KAGGLE={IS_KAGGLE}')
print(f'PROJECT_ROOT={PROJECT_ROOT}')
print(f'DATA_ROOT={DATA_ROOT}')
print(f'OUT_DIR_ECRTM={OUT_DIR_ECRTM}')
print(f'MODEL_TAG={MODEL_TAG}')
print('SPEED_TUNING=', SPEED_TUNING)
print(f'ECRTM_REPO={ECRTM_REPO}')
print(f'FORCE_RECOMPUTE_METRICS={FORCE_RECOMPUTE_METRICS}')



REQUIRED_DATA_FILES = [
    'train_bow.npz',
    'test_bow.npz',
    'train_labels.txt',
    'test_labels.txt',
    'vocab.txt',
    'word_embeddings.npz',
]


def has_required_files(path: Path) -> bool:
    return path.exists() and path.is_dir() and all((path / f).exists() for f in REQUIRED_DATA_FILES)


def guess_dataset_name(path: Path):
    s = str(path).lower()
    if '20ng' in s or '20news' in s:
        return '20NG'
    if 'agnews' in s or 'ag_news' in s:
        return 'AGNews'
    if 'yahoo' in s:
        return 'YahooAnswer'
    if 'imdb' in s:
        return 'IMDB'
    return None


def iter_candidate_dirs(base: Path, max_depth: int = 6):
    if not base.exists():
        return
    base_depth = len(base.parts)
    for root, dirs, _ in os.walk(base):
        root_path = Path(root)
        depth = len(root_path.parts) - base_depth
        if depth > max_depth:
            dirs[:] = []
            continue
        yield root_path


EXPLICIT_KAGGLE_CANDIDATES = {
    '20NG': [
        '/kaggle/input/ectrm-source-data-20ng',
        '/kaggle/input/kaggleinputectrm-source-data-20ng',
        '/kaggle/input/datasets/snlosnl/kaggleinputectrm-source-data-20ng',
    ],
    'AGNews': [
        '/kaggle/input/ECTRM_source/data/AGNews',
        '/kaggle/input/ectrm-sourcedataagnews',
        '/kaggle/input/ectrm-source-data-agnews',
        '/kaggle/input/datasets/snlosnl/ectrm-sourcedataagnews',
    ],
    'IMDB': [
        '/kaggle/input/ectrm-sourcedataIMDB',
        '/kaggle/input/ectrm-sourcedataimdb',
        '/kaggle/input/ectrm-source-data-imdb',
        '/kaggle/input/datasets/snlosnl/ectrm-sourcedataimdb',
    ],
    'YahooAnswer': [
        '/kaggle/input/ECTRM-YahooAnswer',
        '/kaggle/input/ectrm-yahooanswer',
        '/kaggle/input/ectrm-source-data-yahooanswer',
        '/kaggle/input/datasets/snlosnl/ectrm-yahooanswer',
    ],
}


def first_valid_path(candidates):
    for pstr in candidates:
        p = Path(pstr)
        if has_required_files(p):
            return p
    return None


def discover_dataset_dirs():
    found = {}

    if IS_KAGGLE:
        for ds, candidates in EXPLICIT_KAGGLE_CANDIDATES.items():
            p = first_valid_path(candidates)
            if p is not None:
                found[ds] = p

    local_data_root = PROJECT_ROOT / 'ECTRM_source' / 'ECRTM' / 'data'
    for ds in TARGET_DATASETS:
        p = local_data_root / ds
        if has_required_files(p) and ds not in found:
            found[ds] = p

    # Also try PROJECT_ROOT/data style
    for ds in TARGET_DATASETS:
        p = DATA_ROOT / ds
        if has_required_files(p) and ds not in found:
            found[ds] = p

    scan_roots = [Path('/kaggle/input'), PROJECT_ROOT] if IS_KAGGLE else [PROJECT_ROOT]
    seen = set()
    for scan_root in scan_roots:
        for cand in iter_candidate_dirs(scan_root):
            cs = str(cand)
            if cs in seen:
                continue
            seen.add(cs)
            if not has_required_files(cand):
                continue
            ds = guess_dataset_name(cand)
            if ds is not None and ds not in found:
                found[ds] = cand

    return found


DATASET_DIRS = discover_dataset_dirs()
DATASETS = [ds for ds in TARGET_DATASETS if ds in DATASET_DIRS]
if not DATASETS:
    DATASETS = TARGET_DATASETS

print('Resolved datasets:')
for ds in TARGET_DATASETS:
    print('-', ds, DATASET_DIRS.get(ds))


def choose_amp_dtype():
    if not torch.cuda.is_available():
        return None

    if SPEED_TUNING.get('prefer_bf16', True):
        try:
            if hasattr(torch.cuda, 'is_bf16_supported') and torch.cuda.is_bf16_supported():
                return torch.bfloat16
        except Exception:
            pass
    return torch.float16


def make_grad_scaler(enabled: bool):
    # torch.amp API (new) with fallback to torch.cuda.amp for compatibility.
    try:
        return torch.amp.GradScaler('cuda', enabled=enabled)
    except Exception:
        return GradScaler(enabled=enabled)


ECRTM source introuvable localement, tentative de clone GitHub...


Cloning into '/kaggle/working/_ecrtm_source/ecrtm'...


IS_KAGGLE=True
PROJECT_ROOT=/kaggle/working
DATA_ROOT=/kaggle/working/data
OUT_DIR_ECRTM=/kaggle/working/Sortie_mat/output/kaggle/ECRTM_v2_SPEED
MODEL_TAG=ECRTM_v2_SPEED
SPEED_TUNING= {'use_amp': True, 'amp_adaptive_fallback': True, 'prefer_bf16': True, 'num_workers': 2, 'pin_memory': True, 'non_blocking_transfer': True, 'grad_clip': 1.0, 'use_early_stopping': False, 'early_stopping_patience': 12, 'early_stopping_min_delta': 0.0001, 'min_epochs_before_stop': 80, 'restore_best_state': True, 'max_non_finite_batches': 8}
ECRTM_REPO=/kaggle/working/_ecrtm_source/ecrtm/ECRTM
FORCE_RECOMPUTE_METRICS=True
Resolved datasets:
- 20NG /kaggle/input/datasets/snlosnl/kaggleinputectrm-source-data-20ng
- AGNews /kaggle/input/datasets/snlosnl/ectrm-sourcedataagnews
- IMDB /kaggle/input/datasets/snlosnl/ectrm-sourcedataimdb
- YahooAnswer /kaggle/input/datasets/snlosnl/ectrm-yahooanswer


In [3]:
def load_cfg(path):
    with open(path, "r", encoding="utf-8") as f:
        return yaml.safe_load(f)


def load_vocab(path):
    with open(path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]


def load_bow_npz(path):
    X = sp.load_npz(path).astype(np.float32)
    return X.toarray()


def load_word_emb_npz(path):
    W = sp.load_npz(path)
    return W.astype(np.float32).toarray()


In [4]:
def set_seed(seed=42):
    import random

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        # Compromis vitesse/reproductibilite pour la variante speed
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True


def build_args_from_yaml(cfg, K, vocab_size, word_emb):
    cfg = dict(cfg)
    cfg["num_topic"] = K
    cfg["vocab_size"] = vocab_size
    cfg["word_embeddings"] = word_emb
    return SimpleNamespace(**cfg)


def infer_theta(model, bow_np, device, batch_size=256):
    model.eval()
    thetas = []
    with torch.no_grad():
        X = torch.from_numpy(bow_np).float()
        loader = DataLoader(TensorDataset(X), batch_size=batch_size, shuffle=False)
        for (bow,) in loader:
            bow = bow.to(device)
            theta = model.get_theta(bow)
            thetas.append(theta.cpu().numpy())
    return np.vstack(thetas)


In [5]:
def train_one(dataset, K, seed=42):
    set_seed(seed)
    total_start = time.perf_counter()

    if dataset not in DATASET_DIRS:
        raise KeyError(f'Dataset {dataset} introuvable. Disponibles: {list(DATASET_DIRS.keys())}')
    data_dir = DATASET_DIRS[dataset]

    cfg_path = CONFIGS[dataset]
    if not cfg_path.exists():
        raise FileNotFoundError(f"Config introuvable: {cfg_path}")

    cfg = load_cfg(cfg_path)
    epochs = int(cfg["epochs"])
    batch_size = int(cfg["batch_size"])
    lr = float(cfg["learning_rate"])

    train_bow = load_bow_npz(data_dir / "train_bow.npz")
    test_bow = load_bow_npz(data_dir / "test_bow.npz")
    vocab = load_vocab(data_dir / "vocab.txt")
    word_emb = load_word_emb_npz(data_dir / "word_embeddings.npz")

    V = train_bow.shape[1]
    assert V == test_bow.shape[1] == len(vocab) == word_emb.shape[0], "Mismatch vocab sizes"

    args = build_args_from_yaml(cfg, K=K, vocab_size=V, word_emb=word_emb)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    amp_requested = bool(SPEED_TUNING.get('use_amp', True) and device == 'cuda')
    amp_dtype = choose_amp_dtype() if amp_requested else None

    num_workers = int(SPEED_TUNING['num_workers']) if device == 'cuda' else 0
    pin_memory = bool(SPEED_TUNING['pin_memory'] and device == 'cuda')
    non_blocking = bool(SPEED_TUNING['non_blocking_transfer'] and device == 'cuda')

    X = torch.from_numpy(train_bow).float()
    loader = DataLoader(
        TensorDataset(X),
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=pin_memory,
        persistent_workers=(num_workers > 0),
    )

    use_early_stopping = bool(SPEED_TUNING.get('use_early_stopping', False))
    patience = int(SPEED_TUNING['early_stopping_patience'])
    min_delta = float(SPEED_TUNING['early_stopping_min_delta'])
    min_epochs_before_stop = int(SPEED_TUNING['min_epochs_before_stop'])
    grad_clip = float(SPEED_TUNING['grad_clip'])
    max_non_finite_batches = int(SPEED_TUNING.get('max_non_finite_batches', 8))

    def _train_attempt(use_amp_flag: bool):
        model = ECRTM(args).to(device)
        opt = torch.optim.Adam(model.parameters(), lr=lr)
        scaler = make_grad_scaler(enabled=use_amp_flag)

        best_loss = float('inf')
        best_state = None
        bad_epochs = 0
        early_stopped = False
        epochs_ran = 0

        train_start = time.perf_counter()
        epoch_bar = tqdm(range(1, epochs + 1), desc=f"{dataset} | K={K} | seed={seed} | amp={use_amp_flag}")

        for epoch in epoch_bar:
            model.train()
            total_loss = 0.0
            valid_batches = 0
            non_finite_batches = 0

            for (bow,) in loader:
                bow = bow.to(device, non_blocking=non_blocking)
                opt.zero_grad(set_to_none=True)

                if use_amp_flag:
                    try:
                        ctx = torch.amp.autocast('cuda', dtype=amp_dtype)
                    except Exception:
                        ctx = autocast(enabled=True)
                else:
                    ctx = nullcontext()

                with ctx:
                    rst = model(bow)
                    loss = rst["loss"]

                if not torch.isfinite(loss):
                    non_finite_batches += 1
                    if non_finite_batches <= 3:
                        print(f"[WARN] non-finite loss epoch={epoch} batch skipped (amp={use_amp_flag})")
                    if non_finite_batches >= max_non_finite_batches:
                        raise RuntimeError('NON_FINITE_LOSS')
                    continue

                if use_amp_flag:
                    scaler.scale(loss).backward()
                    scaler.unscale_(opt)
                    if grad_clip > 0:
                        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                    scaler.step(opt)
                    scaler.update()
                else:
                    loss.backward()
                    if grad_clip > 0:
                        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                    opt.step()

                total_loss += float(loss.detach().cpu())
                valid_batches += 1

            if valid_batches == 0:
                raise RuntimeError('NON_FINITE_LOSS')

            epochs_ran = epoch
            avg_loss = total_loss / valid_batches
            if not np.isfinite(avg_loss):
                raise RuntimeError('NON_FINITE_LOSS')

            epoch_bar.set_postfix(avg_loss=avg_loss, best_loss=best_loss, amp=use_amp_flag, valid_batches=valid_batches)

            improved = avg_loss + min_delta < best_loss
            if improved:
                best_loss = avg_loss
                bad_epochs = 0
                if SPEED_TUNING['restore_best_state']:
                    best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            else:
                bad_epochs += 1

            if use_early_stopping and epoch >= min_epochs_before_stop and bad_epochs >= patience:
                early_stopped = True
                print(f"[EarlyStopping] stop at epoch={epoch} (best_loss={best_loss:.6f})")
                break

        if SPEED_TUNING['restore_best_state'] and best_state is not None:
            model.load_state_dict(best_state)

        train_seconds = time.perf_counter() - train_start
        return model, {
            'train_seconds': float(train_seconds),
            'epochs_ran': int(epochs_ran),
            'early_stopped': bool(early_stopped),
            'best_loss': float(best_loss) if np.isfinite(best_loss) else float('nan'),
            'amp_used': bool(use_amp_flag),
        }

    amp_fallback_to_fp32 = False
    if amp_requested:
        try:
            model, train_meta = _train_attempt(use_amp_flag=True)
        except RuntimeError as e:
            if SPEED_TUNING.get('amp_adaptive_fallback', True) and 'NON_FINITE_LOSS' in str(e):
                print('[AMP fallback] Non-finite detected with AMP, retrying in full precision...')
                amp_fallback_to_fp32 = True
                model, train_meta = _train_attempt(use_amp_flag=False)
            else:
                raise
    else:
        model, train_meta = _train_attempt(use_amp_flag=False)

    train_seconds = train_meta['train_seconds']

    model.eval()
    with torch.no_grad():
        beta = model.get_beta().cpu().numpy()

    train_theta = infer_theta(model, train_bow, device=device, batch_size=256)
    test_theta = infer_theta(model, test_bow, device=device, batch_size=256)

    out_dir = OUT_DIR_ECRTM / dataset
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"{dataset}_ECRTM_K{K}_seed{seed}.mat"
    scipy.io.savemat(str(out_path), {"beta": beta, "train_theta": train_theta, "test_theta": test_theta})

    total_seconds = time.perf_counter() - total_start
    timing_row = {
        "model": MODEL_TAG, "dataset": dataset, "K": int(K), "seed": int(seed),
        "device": device, "phase": "train",
        "train_seconds": float(train_seconds), "total_seconds": float(total_seconds),
        "train_minutes": float(train_seconds / 60.0), "total_minutes": float(total_seconds / 60.0),
        "epochs_cfg": int(epochs), "epochs_ran": int(train_meta['epochs_ran']),
        "amp_requested": bool(amp_requested), "amp": bool(train_meta['amp_used']),
        "amp_fallback_to_fp32": bool(amp_fallback_to_fp32),
        "amp_dtype": str(amp_dtype) if amp_dtype is not None else 'none',
        "early_stopped": bool(train_meta['early_stopped']),
        "best_loss": float(train_meta['best_loss']) if np.isfinite(train_meta['best_loss']) else None,
    }

    timing_path = out_dir / f"{dataset}_ECRTM_K{K}_seed{seed}_timing.csv"
    pd.DataFrame([timing_row]).to_csv(timing_path, index=False)

    print(f"Saved: {out_path}")
    print(
        f"Timing: {timing_path} | train={train_seconds:.2f}s total={total_seconds:.2f}s "
        f"| amp_requested={amp_requested} amp_used={train_meta['amp_used']} fallback={amp_fallback_to_fp32}"
    )

    return {"mat_path": str(out_path), "timing_path": str(timing_path), "timing": timing_row}


## Fonctions de métriques (identiques au baseline)

On conserve exactement les mêmes métriques et la même logique d'evaluation pour que la comparaison ECRTM baseline vs ECRTM_v2_speed soit valide scientifiquement.


In [6]:
def purity_score(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    purity = 0
    for c in np.unique(y_pred):
        idx = np.where(y_pred == c)[0]
        if len(idx) > 0:
            _, counts = np.unique(y_true[idx], return_counts=True)
            purity += counts.max()
    return purity / max(len(y_true), 1)


_CACHE = {}


def build_tokenized_texts(train_bow, vocab, max_docs=10000):
    texts = []
    for doc in train_bow[:max_docs]:
        idx = np.where(doc > 0)[0]
        words = [vocab[i] for i in idx if i < len(vocab)]
        if len(words) >= 2:
            texts.append(words)
    return texts


def load_gensim_artifacts(dataset):
    if dataset in _CACHE:
        return _CACHE[dataset]

    if dataset not in DATASET_DIRS:
        raise KeyError(f'Dataset {dataset} introuvable dans DATASET_DIRS')
    data_dir = DATASET_DIRS[dataset]
    train_bow = load_bow_npz(data_dir / "train_bow.npz")
    vocab = load_vocab(data_dir / "vocab.txt")
    texts = build_tokenized_texts(train_bow, vocab)
    dictionary = Dictionary(texts)

    _CACHE[dataset] = (vocab, texts, dictionary)
    return _CACHE[dataset]


def compute_cv_gensim(beta, vocab, texts, dictionary, topn=15):
    topics = []
    for k in range(beta.shape[0]):
        top_idx = np.argsort(beta[k])[::-1][:topn]
        topic_words = [
            vocab[i]
            for i in top_idx
            if i < len(vocab) and vocab[i] in dictionary.token2id
        ]
        if len(topic_words) >= 2:
            topics.append(topic_words)

    if not topics:
        return 0.0

    cm = CoherenceModel(
        topics=topics,
        texts=texts,
        dictionary=dictionary,
        coherence="c_v",
    )
    return float(cm.get_coherence())


## Pipeline des métriques

Cette partie relit les `.mat` entraines par la variante speed et calcule C_V, TD, Purity et NMI.


In [7]:
def process_ecrtm_dataset(dataset, K, seed=42):
    if dataset not in DATASET_DIRS:
        print(f'[SKIP] dataset absent: {dataset}')
        return None

    data_dir = DATASET_DIRS[dataset]
    dataset_dir = OUT_DIR_ECRTM / dataset
    dataset_dir.mkdir(parents=True, exist_ok=True)

    mat_filename = f"{dataset}_ECRTM_K{K}_seed{seed}.mat"
    mat_path = dataset_dir / mat_filename
    metrics_path = dataset_dir / f"metrics_{dataset}_K{K}_seed{seed}.csv"

    if metrics_path.exists() and not FORCE_RECOMPUTE_METRICS:
        print(f"[SKIP] Métriques déjà extraites pour {dataset} K={K}")
        return pd.read_csv(metrics_path).to_dict("records")[0]

    if not mat_path.exists():
        print(f"[MISSING] {mat_path}")
        return None

    print(f"[RUN] {mat_filename} -> Calcul des métriques")

    d = scipy.io.loadmat(str(mat_path))
    if "beta" not in d:
        print("[ERROR] 'beta' absent du .mat")
        return None

    beta = np.asarray(d["beta"])
    if beta.shape[0] != K and beta.shape[1] == K:
        beta = beta.T

    theta_key = "test_theta" if "test_theta" in d else "train_theta"
    theta = np.asarray(d.get(theta_key))
    if theta.ndim != 2:
        print("[ERROR] theta invalide")
        return None

    label_file = "test_labels.txt" if theta_key == "test_theta" else "train_labels.txt"
    y_true = np.loadtxt(data_dir / label_file, dtype=int)
    y_pred = theta.argmax(axis=1)

    n = min(len(y_true), len(y_pred))
    y_true = y_true[:n]
    y_pred = y_pred[:n]

    purity = purity_score(y_true, y_pred)
    nmi = float(normalized_mutual_info_score(y_true, y_pred))

    vocab, texts, dictionary = load_gensim_artifacts(dataset)

    topics = []
    for k in range(beta.shape[0]):
        top_idx = np.argsort(beta[k])[::-1][:25]
        topics.append([vocab[i] for i in top_idx if i < len(vocab)])

    all_words = [w for t in topics for w in t]
    td = len(set(all_words)) / max(len(all_words), 1)

    cv = compute_cv_gensim(beta, vocab, texts, dictionary, topn=15)

    res = {
        "Dataset": dataset,
        "K": int(K),
        "Seed": int(seed),
        "C_V": round(cv, 4),
        "TD": round(td, 4),
        "Purity": round(float(purity), 4),
        "NMI": round(float(nmi), 4),
    }

    pd.DataFrame([res]).to_csv(metrics_path, index=False)
    print(f"[OK] {metrics_path.name}")
    return res


## Exécution finale

Ordre recommande: d'abord la cellule d'entraînement (temps), puis cette cellule de métriques.


In [8]:
# Entraînement ECRTM v2 speed + tableau des temps
training_time_rows = []

for dataset in DATASETS:
    if dataset not in DATASET_DIRS:
        print(f'SKIP missing dataset: {dataset}')
        continue
    print(f"\n=== Entraînement ECRTM_v2_speed: {dataset} ===")
    for K in K_VALUES:
        result = train_one(dataset, K, seed=SEED)
        if isinstance(result, dict) and "timing" in result:
            training_time_rows.append(result["timing"])

if training_time_rows:
    df_train_times = pd.DataFrame(training_time_rows).sort_values(["dataset", "K"]).reset_index(drop=True)
    display(df_train_times[["dataset", "K", "device", "train_seconds", "total_seconds", "epochs_ran", "amp_requested", "amp", "amp_fallback_to_fp32", "amp_dtype", "early_stopped"]])

    time_csv = OUT_DIR_ECRTM / "ECRTM_v2_speed_training_times.csv"
    df_train_times.to_csv(time_csv, index=False)
    print(f"Saved training times: {time_csv}")
else:
    print("No training timing rows collected")


=== Entrainement ECRTM_v2_speed: 20NG ===


20NG | K=20 | seed=42 | amp=True:   0%|          | 0/500 [00:00<?, ?it/s]

Saved: /kaggle/working/Sortie_mat/output/kaggle/ECRTM_v2_SPEED/20NG/20NG_ECRTM_K20_seed42.mat
Timing: /kaggle/working/Sortie_mat/output/kaggle/ECRTM_v2_SPEED/20NG/20NG_ECRTM_K20_seed42_timing.csv | train=1081.95s total=1086.47s | amp_requested=True amp_used=True fallback=False


20NG | K=50 | seed=42 | amp=True:   0%|          | 0/500 [00:00<?, ?it/s]

Saved: /kaggle/working/Sortie_mat/output/kaggle/ECRTM_v2_SPEED/20NG/20NG_ECRTM_K50_seed42.mat
Timing: /kaggle/working/Sortie_mat/output/kaggle/ECRTM_v2_SPEED/20NG/20NG_ECRTM_K50_seed42_timing.csv | train=1085.46s total=1086.44s | amp_requested=True amp_used=True fallback=False


...

Saved: /kaggle/working/Sortie_mat/output/kaggle/ECRTM_v2_SPEED/YahooAnswer/YahooAnswer_ECRTM_K50_seed42.mat
Timing: /kaggle/working/Sortie_mat/output/kaggle/ECRTM_v2_SPEED/YahooAnswer/YahooAnswer_ECRTM_K50_seed42_timing.csv | train=977.05s total=977.66s | amp_requested=True amp_used=True fallback=False


YahooAnswer | K=100 | seed=42 | amp=True:   0%|          | 0/500 [00:00<?, ?it/s]

Saved: /kaggle/working/Sortie_mat/output/kaggle/ECRTM_v2_SPEED/YahooAnswer/YahooAnswer_ECRTM_K100_seed42.mat
Timing: /kaggle/working/Sortie_mat/output/kaggle/ECRTM_v2_SPEED/YahooAnswer/YahooAnswer_ECRTM_K100_seed42_timing.csv | train=984.19s total=984.79s | amp_requested=True amp_used=True fallback=False


,dataset,K,device,train_seconds,total_seconds,epochs_ran,amp_requested,amp,amp_fallback_to_fp32,amp_dtype,early_stopped
0,20NG,20,cuda,1081.950969,1086.465781,500,True,True,False,torch.bfloat16,False
1,20NG,50,cuda,1085.460908,1086.437623,500,True,True,False,torch.bfloat16,False
2,20NG,100,cuda,1107.057597,1107.986738,500,True,True,False,torch.bfloat16,False
3,AGNews,20,cuda,965.581268,966.302177,500,True,True,False,torch.bfloat16,False
4,AGNews,50,cuda,959.717608,960.338281,500,True,True,False,torch.bfloat16,False
5,AGNews,100,cuda,961.260684,961.864303,500,True,True,False,torch.bfloat16,False
6,IMDB,20,cuda,2212.619532,2215.047149,500,True,True,False,torch.bfloat16,False
7,IMDB,50,cuda,2387.553788,2389.757893,500,True,True,False,torch.bfloat16,False
8,IMDB,100,cuda,2414.627782,2416.883140,500,True,True,False,torch.bfloat16,False
9,YahooAnswer,20,cuda,970.961364,971.624426,500,True,True,False,torch.bfloat16,False


Saved training times: /kaggle/working/Sortie_mat/output/kaggle/ECRTM_v2_SPEED/ECRTM_v2_speed_training_times.csv


In [9]:
final_summary = []

for dataset in DATASETS:
    if dataset not in DATASET_DIRS:
        print(f'SKIP missing dataset: {dataset}')
        continue
    print(f"\n--- Analyse: {dataset} ---")
    for K in K_VALUES:
        res = process_ecrtm_dataset(dataset, K, seed=SEED)
        if res is not None:
            final_summary.append(res)

if final_summary:
    df_results = pd.DataFrame(final_summary).sort_values(["Dataset", "K"])
    display(df_results)

    out_csv = OUT_DIR_ECRTM / "ECRTM_v2_speed_ALL_METRICS.csv"
    df_results.to_csv(out_csv, index=False)
    print(f"R?sum? sauvegard?: {out_csv}")
else:
    print("Aucun résultat traité.")



--- Analyse: 20NG ---
[RUN] 20NG_ECRTM_K20_seed42.mat -> Calcul des m?triques
[OK] metrics_20NG_K20_seed42.csv
[RUN] 20NG_ECRTM_K50_seed42.mat -> Calcul des m?triques
[OK] metrics_20NG_K50_seed42.csv
[RUN] 20NG_ECRTM_K100_seed42.mat -> Calcul des m?triques
[OK] metrics_20NG_K100_seed42.csv

--- Analyse: AGNews ---
[RUN] AGNews_ECRTM_K20_seed42.mat -> Calcul des m?triques
[OK] metrics_AGNews_K20_seed42.csv
[RUN] AGNews_ECRTM_K50_seed42.mat -> Calcul des m?triques
[OK] metrics_AGNews_K50_seed42.csv
[RUN] AGNews_ECRTM_K100_seed42.mat -> Calcul des m?triques
[OK] metrics_AGNews_K100_seed42.csv

--- Analyse: IMDB ---
[RUN] IMDB_ECRTM_K20_seed42.mat -> Calcul des m?triques
[OK] metrics_IMDB_K20_seed42.csv
[RUN] IMDB_ECRTM_K50_seed42.mat -> Calcul des m?triques
[OK] metrics_IMDB_K50_seed42.csv
[RUN] IMDB_ECRTM_K100_seed42.mat -> Calcul des m?triques
[OK] metrics_IMDB_K100_seed42.csv

--- Analyse: YahooAnswer ---
[RUN] YahooAnswer_ECRTM_K20_seed42.mat -> Calcul des m?triques
[OK] metrics_Yaho

,Dataset,K,Seed,C_V,TD,Purity,NMI
0,20NG,20,42,0.4922,1.0000,0.4392,0.5393
1,20NG,50,42,0.4587,0.9520,0.4595,0.4418
2,20NG,100,42,0.4685,0.9952,0.5437,0.4668
3,AGNews,20,42,0.4577,0.9080,0.4760,0.2186
4,AGNews,50,42,0.5628,0.9832,0.7624,0.3324
5,AGNews,100,42,0.6463,0.9888,0.4952,0.1907
6,IMDB,20,42,0.3794,0.9400,0.6916,0.0744
7,IMDB,50,42,0.4633,0.9928,0.6709,0.0511
8,IMDB,100,42,0.4666,0.9580,0.6766,0.0392
9,YahooAnswer,20,42,0.4964,1.0000,0.5468,0.3087


R?sum? sauvegard?: /kaggle/working/Sortie_mat/output/kaggle/ECRTM_v2_SPEED/ECRTM_v2_speed_ALL_METRICS.csv
